# LouvainNet v3 — Final Verdict
**C-DE422 Social Network Analytics | Spring 2026**

Compares LouvainNet against Newman on two criteria:
- **Determinism** — do repeated runs produce identical outputs?
- **Speed** — Newman once vs LouvainNet run K times, where K = number of distinct Louvain partitions found on this graph (i.e., the coverage cost Louvain would need to match its own solution space)

Requires `louvainnet_data.pkl` and `louvainnet_results_v3.pkl`.

## 1. Setup

In [1]:
import os, time, pickle, warnings, random as _rng
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import community as community_louvain
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from sklearn.cluster import SpectralClustering
from scipy.sparse.linalg import svds

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

SEED   = 42
DEVICE = torch.device('cpu')
torch.manual_seed(SEED)
np.random.seed(SEED)
print('Imports OK')

Imports OK


In [ ]:
with open('louvainnet_data.pkl', 'rb') as f:
    base = pickle.load(f)
with open('louvainnet_results_v3.pkl', 'rb') as f:
    v3 = pickle.load(f)

G                = base['graph']
newman_partition = base['newman_partition']
nodes_list       = base['nodes_list']
nmi_louvain_runs = base['nmi_louvain_runs']
K_NEWMAN         = len(set(newman_partition.values()))

cfg         = v3['config']
EMBED_DIM   = cfg['embed_dim']
SVD_DIM     = cfg['svd_dim']
RESOLUTIONS = cfg['resolutions']
BASE_SEED   = cfg['base_seed']

# Newman labels as flat array for NMI evaluation
y_raw    = np.array([newman_partition[n] for n in nodes_list], dtype=np.int64)
uniq     = np.unique(y_raw)
remap    = {c: i for i, c in enumerate(uniq)}
Y_NEWMAN = np.array([remap[c] for c in y_raw])

print(f'Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
print(f'Newman K: {K_NEWMAN}')
print(f'LouvainNet v3 config — SVD_DIM={SVD_DIM}, EMBED_DIM={EMBED_DIM}, resolutions={RESOLUTIONS}')

## 2. Rebuild Model and Inference Objects
Reconstruct the GNN from the saved state dict, then build the deterministic consensus once.

In [ ]:
class LouvainNet(nn.Module):
    def __init__(self, in_channels=17, hidden=64, embed_dim=EMBED_DIM, dropout=0.3):
        super().__init__()
        self.conv1   = SAGEConv(in_channels, hidden)
        self.conv2   = SAGEConv(hidden, hidden)
        self.conv3   = SAGEConv(hidden, embed_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv3(x, edge_index)
        return F.normalize(x, p=2, dim=1)


model = LouvainNet().to(DEVICE)
model.load_state_dict(v3['model_state_dict'])
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'LouvainNet v3 loaded — {n_params:,} parameters')

In [ ]:
def build_inference_objects(G, resolutions=RESOLUTIONS, base_seed=BASE_SEED, svd_dim=SVD_DIM):
    """Build consensus matrix, SVD features, PyG data, k_baseline, and k_modpeak."""
    nodes = sorted(G.nodes())
    n2i   = {n: i for i, n in enumerate(nodes)}
    N, T  = len(nodes), len(resolutions)

    co_cluster = np.zeros((N, N), dtype=np.float64)
    k_baseline, best_q, k_modpeak = None, -1.0, None
    for gamma in resolutions:
        part = community_louvain.best_partition(G, resolution=gamma, random_state=base_seed)
        k    = len(set(part.values()))
        q    = community_louvain.modularity(part, G)
        if abs(gamma - 1.0) < 1e-9:
            k_baseline = k
        if q > best_q:
            best_q = q; k_modpeak = k
        comms = sorted(set(part.values()))
        c2i   = {c: idx for idx, c in enumerate(comms)}
        ind   = np.zeros((N, len(comms)))
        for node, comm in part.items():
            if node in n2i:
                ind[n2i[node], c2i[comm]] = 1.0
        co_cluster += ind @ ind.T
    co_cluster /= T

    consensus = {(u, v): float(co_cluster[n2i[u], n2i[v]]) for u, v in G.edges()}

    d = min(svd_dim, N - 1)
    U, s, _ = svds(co_cluster, k=d)
    svd_feat = U[:, np.argsort(-s)].astype(np.float32)

    deg = np.array([G.degree(n) for n in nodes], dtype=np.float32)
    x   = np.hstack([svd_feat, (deg / deg.max()).reshape(-1, 1)]).astype(np.float32)

    src, dst, ew = [], [], []
    for (u, v), score in consensus.items():
        i, j = n2i[u], n2i[v]
        src += [i, j]; dst += [j, i]; ew += [score, score]

    pyg = Data(
        x=torch.tensor(x),
        edge_index=torch.tensor([src, dst], dtype=torch.long),
        edge_attr=torch.tensor(ew, dtype=torch.float).unsqueeze(1),
        num_nodes=N
    ).to(DEVICE)

    return pyg, nodes, k_baseline, k_modpeak


print('Building consensus (deterministic, run once)...')
t0 = time.time()
pyg_inf, nodes_inf, K_BASELINE, K_MODPEAK = build_inference_objects(G)
t_consensus = time.time() - t0

with torch.no_grad():
    t_fw = time.time()
    EMBEDDINGS = model(pyg_inf.x, pyg_inf.edge_index).numpy()
    t_fw = time.time() - t_fw

# v3 uses k_modpeak (modularity-peak K) for clustering
K_USE = K_MODPEAK

print(f'Consensus built in {t_consensus:.2f}s')
print(f'GNN forward:       {t_fw*1000:.1f}ms')
print(f'K_BASELINE (res=1.0): {K_BASELINE} | K_MODPEAK (max-Q): {K_MODPEAK} | K_NEWMAN: {K_NEWMAN}')
print(f'=> Using K={K_USE} for SpectralClustering')

## 3. Find K_distinct — Louvain's Solution Space

Run Louvain with N_PROBE different seeds. Count how many genuinely distinct partitions it produces.
This K_distinct is the number of times we must run Louvain to cover its own answer space — and the number of LouvainNet runs we'll use in the speed test.

In [5]:
N_PROBE = 50
print(f'Running Louvain {N_PROBE} times to map its solution space...')
t0 = time.time()

louvain_partitions = []
for seed in range(N_PROBE):
    part = community_louvain.best_partition(G, random_state=seed)
    louvain_partitions.append(np.array([part[n] for n in nodes_inf]))

t_probe = time.time() - t0

# Two partitions are the same iff ARI = 1.0 (identical up to label permutation)
distinct_idxs = [0]
for i in range(1, N_PROBE):
    is_new = all(
        adjusted_rand_score(louvain_partitions[j], louvain_partitions[i]) < 0.9999
        for j in distinct_idxs
    )
    if is_new:
        distinct_idxs.append(i)

K_DISTINCT = len(distinct_idxs)
print(f'{N_PROBE} Louvain runs in {t_probe:.1f}s')
print(f'Distinct partitions found: {K_DISTINCT} (seeds: {distinct_idxs})')
print(f'=> Speed test will run LouvainNet {K_DISTINCT} times')

Running Louvain 50 times to map its solution space...


50 Louvain runs in 56.1s
Distinct partitions found: 46 (seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 31, 32, 33, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 47, 48])
=> Speed test will run LouvainNet 46 times


## 4. Determinism Test

In [ ]:
N_REPS = 20

# ── LouvainNet: repeat spectral clustering on fixed embeddings ──
def run_louvainnet_once(embeddings, k, seed=SEED):
    sc = SpectralClustering(n_clusters=k, affinity='rbf', gamma=1.0,
                            random_state=seed, n_init=5)
    return sc.fit_predict(embeddings)

lnet_preds = [run_louvainnet_once(EMBEDDINGS, K_USE) for _ in range(N_REPS)]
lnet_all_same  = all(np.array_equal(lnet_preds[0], lnet_preds[i]) for i in range(1, N_REPS))
lnet_nmis      = [normalized_mutual_info_score(Y_NEWMAN, p) for p in lnet_preds]
lnet_nmi_std   = float(np.std(lnet_nmis))

# ── Newman: repeat with different Python RNG seeds ──
newman_preds = []
for s in range(N_REPS):
    _rng.seed(s); np.random.seed(s)
    comms = list(nx.community.greedy_modularity_communities(G))
    part  = {node: i for i, c in enumerate(comms) for node in c}
    newman_preds.append(np.array([part[n] for n in nodes_inf]))

newman_all_same = all(
    adjusted_rand_score(newman_preds[0], newman_preds[i]) > 0.9999
    for i in range(1, N_REPS)
)
newman_nmi_std = float(np.std([normalized_mutual_info_score(Y_NEWMAN, p) for p in newman_preds]))

print('=== DETERMINISM TEST ===')
print(f'LouvainNet v3 — {N_REPS} runs all identical: {lnet_all_same}  | NMI std: {lnet_nmi_std:.6f}')
print(f'Newman        — {N_REPS} runs all identical: {newman_all_same} | NMI std: {newman_nmi_std:.6f}')

## 5. Speed Test

**Newman** timed once (its natural usage — one deterministic call).

**LouvainNet** timed for K_distinct complete pipeline runs:
- Consensus is built once (shared across all runs — deterministic, same result every time)
- GNN forward + spectral clustering repeated K_distinct times
- Total = `t_consensus + K_distinct × (t_forward + t_cluster)`

This is the *conservative* comparison: even if you needed K_distinct LouvainNet calls to cover the same solution space that Louvain explores in K_distinct runs, is LouvainNet still faster than Newman once?

In [ ]:
# ── Newman: single run ──
_rng.seed(SEED); np.random.seed(SEED)
t0 = time.time()
_ = list(nx.community.greedy_modularity_communities(G))
t_newman_single = time.time() - t0

# ── LouvainNet: K_distinct runs ──
# Consensus already built (t_consensus measured above).
# Time K_distinct forward + cluster calls.
t_loop = time.time()
for _ in range(K_DISTINCT):
    with torch.no_grad():
        emb = model(pyg_inf.x, pyg_inf.edge_index).numpy()
    sc = SpectralClustering(n_clusters=K_USE, affinity='rbf', gamma=1.0,
                            random_state=SEED, n_init=5)
    sc.fit_predict(emb)
t_lnet_loop = time.time() - t_loop

# Total LouvainNet cost = one-time consensus + K_distinct × (forward + cluster)
t_lnet_total  = t_consensus + t_lnet_loop
t_lnet_single = t_consensus + t_lnet_loop / K_DISTINCT  # single run for reference

print('=== SPEED TEST ===')
print(f'Newman             (1 run)             : {t_newman_single:.3f}s')
print(f'LouvainNet v3      (1 full run)         : {t_lnet_single:.3f}s')
print(f'LouvainNet v3      ({K_DISTINCT} runs, K_distinct): {t_lnet_total:.3f}s')
print()
if t_lnet_total < t_newman_single:
    ratio = t_newman_single / t_lnet_total
    print(f'LouvainNet x{K_DISTINCT} is {ratio:.2f}x FASTER than Newman x1')
else:
    ratio = t_lnet_total / t_newman_single
    print(f'LouvainNet x{K_DISTINCT} is {ratio:.2f}x SLOWER than Newman x1')

## 6. Final Verdict

In [ ]:
lnet_nmi = float(np.mean(lnet_nmis))
newman_nmi = 1.0  # Newman vs itself

print('=' * 58)
print('FINAL VERDICT — LouvainNet v3 vs Newman (ego-Facebook)')
print('=' * 58)
print()

print('[ DETERMINISM ]')
print(f'  Newman       : deterministic = {newman_all_same}')
print(f'  LouvainNet v3: deterministic = {lnet_all_same}  (std={lnet_nmi_std:.6f})')
print()

print('[ QUALITY ]')
print(f'  Newman NMI vs self     : 1.0000  (K={K_NEWMAN})')
print(f'  LouvainNet v3 NMI      : {lnet_nmi:.4f}  (K={K_USE}, modpeak | baseline={K_BASELINE})')
print()

print('[ SPEED ]')
print(f'  Newman (1 run)                  : {t_newman_single:.2f}s')
print(f'  LouvainNet v3 (1 run, full)     : {t_lnet_single:.2f}s')
print(f'  LouvainNet v3 ({K_DISTINCT} runs, K_distinct): {t_lnet_total:.2f}s')
if t_lnet_total < t_newman_single:
    print(f'  => LouvainNet x{K_DISTINCT} still {t_newman_single/t_lnet_total:.2f}x faster than Newman x1')
else:
    print(f'  => LouvainNet x{K_DISTINCT} is {t_lnet_total/t_newman_single:.2f}x slower than Newman x1')
print()

verdict_data = {
    'Algorithm'        : ['Newman', f'LouvainNet v3 (x{K_DISTINCT})'],
    'Deterministic'    : [str(newman_all_same), str(lnet_all_same)],
    'NMI vs Newman'    : [f'{newman_nmi:.4f}', f'{lnet_nmi:.4f}'],
    'Time (this test)' : [f'{t_newman_single:.2f}s', f'{t_lnet_total:.2f}s'],
    'K detected'       : [str(K_NEWMAN), f'{K_USE} (modpeak)'],
}
print(pd.DataFrame(verdict_data).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
clrs   = ['#0F6E56', '#534AB7']
labels = ['Newman', f'LouvainNet v3\n(x{K_DISTINCT})']

# Determinism: NMI std across N_REPS runs
std_vals = [newman_nmi_std, lnet_nmi_std]
bars = axes[0].bar(labels, std_vals, color=clrs, edgecolor='white', alpha=0.85)
axes[0].set_ylabel('NMI std across runs (lower = more deterministic)')
axes[0].set_title('Determinism', fontweight='bold')
for bar, v in zip(bars, std_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + max(std_vals + [1e-7])*0.02,
                 f'{v:.6f}', ha='center', fontsize=9)

# Quality: NMI vs Newman
nmi_vals = [newman_nmi, lnet_nmi]
bars2 = axes[1].bar(labels, nmi_vals, color=clrs, edgecolor='white', alpha=0.85)
axes[1].set_ylim(0, 1.2)
axes[1].set_ylabel('NMI vs Newman')
axes[1].set_title('Quality', fontweight='bold')
for bar, v in zip(bars2, nmi_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.02, f'{v:.4f}', ha='center', fontsize=9)

# Speed
time_vals = [t_newman_single, t_lnet_total]
bars3 = axes[2].bar(labels, time_vals, color=clrs, edgecolor='white', alpha=0.85)
axes[2].set_ylabel('Wall-clock time (s)')
axes[2].set_title(f'Speed  (LouvainNet = {K_DISTINCT} full runs)', fontweight='bold')
for bar, v in zip(bars3, time_vals):
    axes[2].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + max(time_vals)*0.01,
                 f'{v:.2f}s', ha='center', fontsize=9)

plt.suptitle('LouvainNet v3 — Final Verdict vs Newman', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('fig_verdict_v3.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_verdict_v3.png')